# Mini-Challenge 1.3

**Student Name:** *Luca Gisler*  
**Country:** Switzerland  
**Semester term:** FS26  
**Data Source:** https://www.kaggle.com/datasets/mohamedasem318/wesad-full-dataset  
**Code:** https://github.com/schwitzkasten/fhnw-ds-gbsv

## Day 11 — Data & Domain

### Use Case
Swiss Rega medics still rely on Empatica E4 BVP telemetry to judge whether a climber developing altitude sickness needs an immediate evacuation. The time-domain convolution stage must damp rope-slap motion artifacts (20–40 Hz) and cabin vibration while retaining the systolic crest and dicrotic notch that encode heart-rate variability. The matching deconvolution stage aims to reverse the deliberately applied smoothing so that any attenuated peak amplitude and timing cues re-emerge before the trace is relayed to dispatchers.

### Problem Statement
Convolution that is not physiologically grounded can flatten cardiac morphology, while deconvolution that ignores noise statistics can explode high-frequency jitter. The unresolved question is whether a compact, pulse-width-scaled kernel can suppress nuisance energy without erasing the ≈0.94 s cardiac cadence, and whether the inverse filter can recover systolic peak amplitude and notch timing after the smoothing stage. Without this two-part workflow, Rega crews must eyeball noisy traces and risk dispatch delays.

### Experimental Objective
1. **Convolution objective.** Derive and apply a short, normalized kernel (≤80 ms support) that lowers high-frequency variance by ≥6 dB yet keeps peak spacing and amplitudes within clinically tolerable error bands.  
2. **Deconvolution objective.** Treat the filtered output as a blurred signal and run a noise-aware inverse (Wiener) operator that restores systolic crest amplitude and the dicrotic notch timing within ±20 ms, enabling downstream heart-rate estimation to stay trustworthy.

### Data Definition, Source, and Visualization Plan
- **Signal + reuse note:** Same detrended 30 s (1,920 samples) BVP window from WESAD/S2 loaded in MC1.1–1.2; no reselection occurs.  
- **Sampling + physical anchors:** 64 Hz sampling (15.625 ms/sample); amplitude in Empatica a.u.; time plotted in seconds.  
- **Structures of interest:** systolic crest spans ≈70 ms (4–5 samples) with a secondary dicrotic notch ≈120 ms later; respiration modulates the envelope near 0.3 Hz.  
- **Visualization:** The Day 13 plots will overlay original, filtered, and reconstructed signals plus the kernel profile so later days can cite exact shapes without replotting.

### Observations
- Rope-slap artifacts insert >15 Hz chatter, yet the pulse train remains quasi-periodic with dominant peaks every 0.94 s, confirming that short kernels can denoise without smearing multiple beats.  
- The systolic crest height varies by 60–70 a.u. across the window, so over-smoothing would bias amplitude-based tachy detection.  
- The dicrotic notch is already shallow; any kernel wider than 5 samples risks erasing it, hence the insistence on pulse-width-derived parameters.  
- Respiratory modulation (~0.3 Hz) acts as a slow envelope, making time-domain convolution preferable to FFT magnitude gating because it preserves instantaneous phase required for beat timing.  
- Deconvolution must account for the Empatica sensor’s quantization noise; otherwise, ringing could create false peaks that medics might misinterpret as premature ventricular contractions.

## Day 12 — Methodological Design

### Theoretical Foundation & Method Choice
Time-domain convolution is preferred over FFT-domain filtering because the cardiac waveform is nonstationary across the 30 s traverse; preserving absolute timing of the ≈0.94 s peaks requires linear, shift-invariant operations with compact support. By anchoring kernels to the 15.625 ms sampling step, we maintain direct control over how many physiological structures fall inside the smoothing window. Deconvolution is framed as an inverse problem where the smoothed output equals the original signal convolved with a known kernel plus additive noise; this aligns with the Empatica pipeline, which already records calibration kernels for in-device smoothing.

### Kernel Selection & Parameter Justification
- **Five-sample moving average.** Pulse width is ≈70 ms, derived from photoplethysmography literature and verified in MC1.2. Dividing by the 15.625 ms sampling period yields 4.5 samples, so an odd-sized 5-sample window (78 ms) spans one systolic crest while keeping the dicrotic notch largely outside the core weights.  
- **Seven-sample Gaussian ($\sigma=1.0$ sample).** A Gaussian with $\sigma=1$ sample (15.6 ms) and length $2\lceil 3\sigma\rceil+1 = 7$ samples retains 99.7% of its energy within 93.8 ms. This covers the crest plus the onset of the notch but leaves adjacent beats untouched because 93.8 ms ≪ 0.94 s. Cascading the moving average and Gaussian emulates Empatica’s firmware (boxcar + Gaussian) while keeping total effective width under 160 ms (<18% of an inter-beat interval).

### Deconvolution Approach Selection
A naive inverse filter would divide the FFT of the blurred signal by the FFT of the composite kernel, but the Empatica BVP noise floor (quantization + sensor jitter) would be amplified where the kernel spectrum approaches zero. A Wiener deconvolution, defined as $\hat{X}(f)=\frac{H^*(f)}{|H(f)|^2+K}Y(f)$, adds a tunable noise-to-signal ratio $K$ that suppresses divisions by small magnitudes while still restoring attenuated peaks. Tikhonov regularization was considered, yet it requires selecting a derivative operator and offers less interpretability for medics; Wiener filtering keeps the adjustable parameter directly tied to the expected noise variance.

### Parameter Sweep Design for Day 14
Baseline configuration: 5-sample moving average → 7-sample Gaussian with $\sigma=1.0$ sample → Wiener deconvolution with $K=10^{-2}$. Swept parameters target physiologically meaningful extremes:  
1. **Kernel size ($N$ in samples):** $N\in\{3,5,9\}$. The 3-sample case (46.9 ms) is an aggressive power-saving filter, 5 samples matches the crest, and 9 samples (140.6 ms) probes the limit before the notch is lost.  
2. **Gaussian $\sigma$ (samples):** $\{0.6,1.0,1.4\}$, equating to energy spans of 56, 94, and 132 ms, respectively, which bracket the systolic crest width.  
3. **Wiener $K$ (unitless noise-to-signal ratio):** $\{10^{-3},10^{-2},10^{-1}\}$, covering calm telemetry (low noise), helicopter rotor contamination (baseline), and severe jitter (high noise).  
These ranges ensure Day 14 can report how much smoothing and inversion aggressiveness the alpine scenario tolerates.

### Limitations & Risk Factors
- **Over-smoothing:** Kernels >9 samples or $\sigma>1.4$ would merge systolic peaks with respiratory envelopes, biasing heart-rate estimation.  
- **Noise amplification:** Too-small $K$ in the Wiener stage (<$10^{-3}$) can boost quantization noise into visible ringing, creating false peaks.  
- **Boundary artifacts:** Zero-padding during manual convolution attenuates the first and last two beats; Day 13 will mirror-verify via NumPy to ensure the padding choice is disclosed.  
- **Model mismatch:** Assuming linear shift invariance ignores motion-induced baseline wander, so deconvolution might overcorrect segments with saturations, a risk medics must factor in.